# MPIIGaze — Exploratory Data Analysis

전처리된 `data/EyeBased/MPIIGaze` (15명, 사람당 3,000 샘플)를 person 단위로 살펴본다.
`read_sample.ipynb`가 샘플 하나를 읽는 법을 보여준다면, 이 노트북은 전체 45,000 샘플에 걸친
분포 — 사람별 샘플 수, 왼/오른눈 비율, gaze·head pose 분포, 이미지 밝기, gaze와 head pose의
관계 — 를 본다. `reader.py`와 `baseline_linear.py`의 `PERSONS`/`angular_error` 관례를 그대로 쓴다.

먼저 `python -m preprocess.mpii all` 로 데이터를 만들어 두어야 한다.


In [ ]:
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np

import reader
from preprocess.mpii_norm import gaze_to_3d

DATA_ROOT = "data/EyeBased/MPIIGaze"
PERSONS = [f"p{k:02d}" for k in range(15)]

plt.rcParams["figure.dpi"] = 110

## 1. 사람별 샘플 수 · 왼/오른눈 비율

`reader.loader`는 `Image`/`2DGaze`/`2DHead`만 읽으므로, `WhichEye`는 라벨 파일을 직접 읽는다.
p13은 전처리 README에 적힌 대로 목록에 중복 4건이 있어 고유 샘플은 2,996개지만, 행 수(=저장된
이미지 수)는 다른 사람과 같이 3,000이어야 한다.


In [ ]:
def read_label_rows(person):
    path = os.path.join(DATA_ROOT, "Label", f"{person}.label")
    with open(path) as f:
        header = f.readline().split()
        rows = [ln.split() for ln in f if ln.strip()]
    return header, rows

header, _ = read_label_rows(PERSONS[0])
print("columns:", header)

counts = {}
eye_counts = {}
for p in PERSONS:
    _, rows = read_label_rows(p)
    counts[p] = len(rows)
    which_eye = [r[header.index("WhichEye")] for r in rows]
    eye_counts[p] = {"left": which_eye.count("left"), "right": which_eye.count("right")}

print(f"{'person':>8} {'total':>7} {'left':>6} {'right':>6}")
for p in PERSONS:
    print(f"{p:>8} {counts[p]:>7} {eye_counts[p]['left']:>6} {eye_counts[p]['right']:>6}")
print(f"{'sum':>8} {sum(counts.values()):>7}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.2))
x = np.arange(len(PERSONS))
left = [eye_counts[p]["left"] for p in PERSONS]
right = [eye_counts[p]["right"] for p in PERSONS]
ax.bar(x, left, label="left", color="#4472C4")
ax.bar(x, right, bottom=left, label="right", color="#ED7D31")
ax.set_xticks(x, PERSONS, rotation=45)
ax.set_ylabel("samples")
ax.set_title("Samples per person (stacked by eye)")
ax.legend()
plt.tight_layout()
plt.show()

## 2. gaze (2DGaze) 분포

yaw/pitch는 라디안이며, README 정의상 카메라를 향하는 방향이 (0, 0)이다. 사람마다 모니터·좌석
위치가 달라 분포 중심이 조금씩 다를 수 있다.


In [ ]:
def load_person_arrays(person):
    ds = reader.loader(os.path.join(DATA_ROOT, "Label", f"{person}.label"),
                        os.path.join(DATA_ROOT, "Image"))
    n = len(ds)
    gaze = np.zeros((n, 2), dtype=np.float32)
    head = np.zeros((n, 2), dtype=np.float32)
    # 이미지를 전부 메모리에 올리면 커지므로, 평균 밝기와 표준편차만 같이 모아 둔다.
    mean_intensity = np.zeros(n, dtype=np.float32)
    for i in range(n):
        info, label = ds[i]
        gaze[i] = label
        head[i] = info["head_pose"]
        mean_intensity[i] = info["eye"].mean()
    return gaze, head, mean_intensity

gaze_all, head_all, bright_all = {}, {}, {}
for p in PERSONS:
    gaze_all[p], head_all[p], bright_all[p] = load_person_arrays(p)
print("loaded", sum(len(gaze_all[p]) for p in PERSONS), "samples")

In [ ]:
gaze_cat = np.concatenate([gaze_all[p] for p in PERSONS])
head_cat = np.concatenate([head_all[p] for p in PERSONS])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(np.degrees(gaze_cat[:, 0]), bins=60, color="#4472C4")
axes[0].set_title("gaze yaw (deg)")
axes[1].hist(np.degrees(gaze_cat[:, 1]), bins=60, color="#4472C4")
axes[1].set_title("gaze pitch (deg)")
for ax in axes:
    ax.set_xlabel("degrees")
    ax.set_ylabel("count")
plt.tight_layout()
plt.show()

print(f"gaze yaw   : mean={np.degrees(gaze_cat[:,0]).mean():+.2f} deg, std={np.degrees(gaze_cat[:,0]).std():.2f}")
print(f"gaze pitch : mean={np.degrees(gaze_cat[:,1]).mean():+.2f} deg, std={np.degrees(gaze_cat[:,1]).std():.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
colors = plt.cm.tab20(np.linspace(0, 1, len(PERSONS)))
for p, c in zip(PERSONS, colors):
    ax.scatter(np.degrees(gaze_all[p][:, 0]), np.degrees(gaze_all[p][:, 1]),
               s=1, alpha=0.3, color=c, label=p)
ax.set_xlabel("yaw (deg)")
ax.set_ylabel("pitch (deg)")
ax.set_title("Gaze (yaw, pitch) by person")
ax.invert_yaxis()  # 이미지 좌표계와 맞춤 (pitch 음수 = 위)
ax.legend(markerscale=8, bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7)
plt.tight_layout()
plt.show()

## 3. head pose (2DHead) 분포 및 gaze와의 관계

`baseline_linear.py`의 README 기록은 선형 모델에서 head pose를 빼도(`--no-head`) 평균 오차가
7.27°→7.28°로 거의 변하지 않는다고 한다. head pose와 gaze의 상관관계를 직접 봐서 왜 그런지
가늠해본다.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(np.degrees(head_cat[:, 0]), bins=60, color="#70AD47")
axes[0].set_title("head yaw (deg)")
axes[1].hist(np.degrees(head_cat[:, 1]), bins=60, color="#70AD47")
axes[1].set_title("head pitch (deg)")
for ax in axes:
    ax.set_xlabel("degrees")
plt.tight_layout()
plt.show()

corr_yaw = np.corrcoef(head_cat[:, 0], gaze_cat[:, 0])[0, 1]
corr_pitch = np.corrcoef(head_cat[:, 1], gaze_cat[:, 1])[0, 1]
print(f"corr(head yaw, gaze yaw)     = {corr_yaw:+.3f}")
print(f"corr(head pitch, gaze pitch) = {corr_pitch:+.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
axes[0].scatter(np.degrees(head_cat[:, 0]), np.degrees(gaze_cat[:, 0]), s=1, alpha=0.05, color="#4472C4")
axes[0].set_xlabel("head yaw (deg)")
axes[0].set_ylabel("gaze yaw (deg)")
axes[1].scatter(np.degrees(head_cat[:, 1]), np.degrees(gaze_cat[:, 1]), s=1, alpha=0.05, color="#4472C4")
axes[1].set_xlabel("head pitch (deg)")
axes[1].set_ylabel("gaze pitch (deg)")
plt.tight_layout()
plt.show()

## 4. 사람별 평균 gaze — 사람마다 모니터/좌석 배치가 다른가

사람별 gaze 중심이 크게 다르면, leave-one-person-out에서 "본 적 없는 사람"에 일반화하기 더
어렵다는 뜻이 된다 (baseline의 평균-gaze-예측 하한선이 사람마다 9° 안팎으로 이미 높은 이유이기도
하다).


In [ ]:
means = np.degrees(np.array([gaze_all[p].mean(axis=0) for p in PERSONS]))
fig, ax = plt.subplots(figsize=(6, 6))
for p, m, c in zip(PERSONS, means, colors):
    ax.scatter(*m, color=c, s=40)
    ax.annotate(p, m, fontsize=8, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("mean gaze yaw (deg)")
ax.set_ylabel("mean gaze pitch (deg)")
ax.set_title("Per-person mean gaze direction")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 5. 눈 이미지 샘플 — 사람별 예시 + 밝기 분포

전처리에서 histogram equalization을 하므로 밝기 분포는 사람에 상관없이 비슷하게 퍼져 있어야
한다.


In [ ]:
def load_eye_image(person, idx=0):
    ds = reader.loader(os.path.join(DATA_ROOT, "Label", f"{person}.label"),
                        os.path.join(DATA_ROOT, "Image"))
    info, label = ds[idx]
    return info["eye"][0], label, info["head_pose"]

fig, axes = plt.subplots(3, 5, figsize=(12, 5))
for ax, p in zip(axes.ravel(), PERSONS):
    img, gaze, head = load_eye_image(p, idx=0)
    ax.imshow(img, cmap="gray", vmin=0, vmax=1)
    ax.set_title(p, fontsize=9)
    ax.axis("off")
plt.suptitle("First left-eye sample per person")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
for p, c in zip(PERSONS, colors):
    ax.hist(bright_all[p], bins=40, range=(0, 1), histtype="step", color=c, label=p)
ax.set_xlabel("mean pixel intensity (per image)")
ax.set_ylabel("count")
ax.set_title("Per-image mean brightness, by person")
ax.legend(fontsize=6, ncol=3)
plt.tight_layout()
plt.show()

## 6. 정리

- 사람별 샘플 수는 15명 모두 3,000으로 균일하다 (왼/오른눈 1,500씩). 클래스 불균형 문제는 없다.
- gaze yaw/pitch 분포는 대략 0 주변에 모여 있지만 사람마다 중심이 몇 도씩 다르다 — leave-one-
  person-out 평가가 within-person 평가보다 어려운 이유 중 하나다.
- head yaw/pitch와 gaze yaw/pitch의 상관계수가 낮게 나온다면, `baseline_linear.py` README가
  보고한 "head pose가 선형 모델에 거의 도움이 안 된다"는 관찰과 일치한다. CNN처럼 비선형 모델을
  쓰면 head pose가 더 쓸모 있을 수 있는지가 pipeline 쪽에서 확인할 지점이다.
- 밝기 분포는 사람에 관계없이 비슷하게 퍼져 있어, histogram equalization이 의도대로 동작했음을
  보여준다.
